# Import and setup

In [ ]:
import os, torch, torch.nn as nn, torch.optim as optim, pandas as pd, numpy as np
import time
import matplotlib.pyplot as plt, seaborn as sns
import glob, librosa
import timm
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tabulate import tabulate
from IPython.display import clear_output
import torchaudio, torchaudio.transforms as T
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('No GPU detected')

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.44 GB


In [3]:
%cd D:\project\apisacoustic

D:\project\apisacoustic


In [4]:
csv_frdr = r"dataset\urban-frdr\inspections_2022.csv"
csv_zenodo = r"dataset\zenodo\state_labels.csv"
frdr_audiofolder_path = r"dataset/urban-frdr/audio_2022_all_chunk"
zenodo_audiofolder_path = r"dataset\zenodo"

### checking status

In [5]:
df_frdr = pd.read_csv(csv_frdr)
target = df_frdr[(df_frdr['Queen status'] == 'queenless') | (df_frdr['Category'] == 'varroa')]
# print(target[['Date', 'Tag number', 'Category', 'Queen status']])

### check status on downloaded chuck

In [6]:
all_files = os.listdir(frdr_audiofolder_path)
extract_list = []
for file_path in all_files:
    if file_path.endswith('.wav'):
        file_name = file_path.split('/')[-1]
        parts = file_name.split('_')
        date = parts[0]
        hive_part = parts[2]
        hive_id = hive_part.split('-')[1].split('.')[0]
        for_merge = pd.to_datetime(date, format='%d-%m-%Y').strftime('%Y-%m-%d')
        extract_list.append({'Filename': file_name, 'Date': for_merge, 'Tag number': int(hive_id)})

df_audio = pd.DataFrame(extract_list)
df_audio['Tag number'] = df_audio['Tag number'].astype(int)

df_frdr['Tag number'] = df_frdr['Tag number'].astype(int)
df_frdr['Date'] = pd.to_datetime(df_frdr['Date']).dt.strftime('%Y-%m-%d')

df_frdr_sorted = df_frdr.sort_values(by=['Date', 'Tag number', 'Category'], ascending=[True, True, False])
df_frdr_clean = df_frdr_sorted.drop_duplicates(subset=['Date', 'Tag number'], keep='first')
merge_df = pd.merge(df_audio, df_frdr_clean, on=['Date', 'Tag number'], how='inner')

def check_class(row):
    if row['Category'] == 'varroa':
        return 'Class 2: Infested'
    elif row['Queen status'] == 'queenless':
        return 'Class 1: Queenless'
    elif row['Queen status'] == 'queenright' and row['Is alive'] == 1:
        return 'Class 0: Active'
    else:
        return 'Others / Ignored'
merge_df['target_Class'] = merge_df.apply(check_class, axis=1)
print(merge_df['target_Class'].value_counts())

target_Class
Class 2: Infested     162
Class 0: Active       147
Class 1: Queenless     83
Name: count, dtype: int64


# Data split

In [7]:
def split_data(df, stratify_col):
    if stratify_col in df.columns:
        stratify_vals = df[stratify_col]
    else:
        stratify_vals = None
    if stratify_vals is not None:
        counts = stratify_vals.value_counts()
        if counts.min() < 2:
            stratify_vals = None
    if len(df) < 10:
        if stratify_vals is not None:
            train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=stratify_vals)
        else:
            train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
        if stratify_vals is not None and stratify_col in temp_df.columns and temp_df[stratify_col].nunique() > 1:
            val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df[stratify_col])
        else:
            val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
        return train_df, val_df, test_df
    if stratify_vals is not None:
        train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=stratify_vals)
    else:
        train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
    if stratify_vals is not None and stratify_col in temp_df.columns:
        counts_temp = temp_df[stratify_col].value_counts()
        if counts_temp.min() < 2:
            val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
        else:
            val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df[stratify_col])
    else:
        val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
    return train_df, val_df, test_df

In [8]:
frdr_files = glob.glob(os.path.join(frdr_audiofolder_path, "*.wav"))
frdr_data = []
for path in frdr_files:
    filename = os.path.basename(path)
    if any(suffix in filename for suffix in ["_Active", "_Queenless", "_Infested"]):
        name_part, _ = os.path.splitext(filename)
        parts = name_part.split('_')
        class_label = parts[-1]
        hive_id = "Unknown"
        for p in parts:
            if "HIVE" in p: hive_id = p; break
        frdr_data.append({'File_Path': path, 'Filename': filename, 'Dataset': 'FRDR', 'Class': class_label, 'Hive': hive_id})
df_frdr_all = pd.DataFrame(frdr_data)

zenodo_files = glob.glob(os.path.join(zenodo_audiofolder_path, "*", "*.wav"))
zenodo_data = []
for path in zenodo_files:
    filename = os.path.basename(path)
    folder_name = os.path.basename(os.path.dirname(path))
    if "NO_QueenBee" in filename: class_label = "Queenless"
    elif "QueenBee" in filename: class_label = "Active"
    else: continue
    hive_id = folder_name.split('_')[0]
    zenodo_data.append({'File_Path': path, 'Filename': filename, 'Dataset': 'ZENODO', 'Class': class_label, 'Hive': hive_id})
df_zenodo_all = pd.DataFrame(zenodo_data)

print(f"UrBAN-FRDR: {len(df_frdr_all)} Files")
print(f"Zenodo: {len(df_zenodo_all)} Files")

zenodo_train_list, zenodo_val_list, zenodo_test_list = [], [], []
for folder, group in df_zenodo_all.groupby('Dataset'):
    z_train, z_val, z_test = split_data(group, stratify_col='Class')
    zenodo_train_list.append(z_train); zenodo_val_list.append(z_val); zenodo_test_list.append(z_test)
train_zenodo = pd.concat(zenodo_train_list) if len(zenodo_train_list) > 0 else pd.DataFrame()
val_zenodo = pd.concat(zenodo_val_list) if len(zenodo_val_list) > 0 else pd.DataFrame()
test_zenodo = pd.concat(zenodo_test_list) if len(zenodo_test_list) > 0 else pd.DataFrame()

df_frdr_all['Stratify_Key'] = df_frdr_all['Class'] + "_" + df_frdr_all['Hive']
counts = df_frdr_all['Stratify_Key'].value_counts()
rare_keys = counts[counts < 10].index
df_frdr_all['Stratify_Key'] = df_frdr_all['Stratify_Key'].apply(lambda x: x.split('_')[0] if x in rare_keys else x)
train_frdr, val_frdr, test_frdr = split_data(df_frdr_all, stratify_col='Stratify_Key')

train_df = pd.concat([train_zenodo, train_frdr], ignore_index=True)
val_df = pd.concat([val_zenodo, val_frdr], ignore_index=True)
test_df = pd.concat([test_zenodo, test_frdr], ignore_index=True)
for df in [train_df, val_df, test_df]:
    df.drop(columns=['Stratify_Key'], errors='ignore', inplace=True)

print(f"Total files: Train = {len(train_df)} | Val = {len(val_df)} | Test = {len(test_df)}")
print("\n")
print("1. Split by Dataset:")
summary_dataset = pd.DataFrame({'Train_Set': train_df['Dataset'].value_counts(), 'Val_Set': val_df['Dataset'].value_counts(), 'Test_Set': test_df['Dataset'].value_counts()}).fillna(0).astype(int)
summary_dataset['Total'] = summary_dataset.sum(axis=1)
print(summary_dataset)
print("\n")
print("2. Split by Class:")
summary_class = pd.DataFrame({'Train_Set': train_df['Class'].value_counts(), 'Val_Set': val_df['Class'].value_counts(), 'Test_Set': test_df['Class'].value_counts()}).fillna(0).astype(int)
summary_class['Total'] = summary_class.sum(axis=1)
print(summary_class)
print("\n")
print("3. Split by Hive:")
summary_hive = pd.DataFrame({'Train_Set': train_df['Hive'].value_counts(), 'Val_Set': val_df['Hive'].value_counts(), 'Test_Set': test_df['Hive'].value_counts()}).fillna(0).astype(int)
summary_hive['Total'] = summary_hive.sum(axis=1)
print(summary_hive)

UrBAN-FRDR: 393 Files
Zenodo: 576 Files
Total files: Train = 774 | Val = 97 | Test = 98


1. Split by Dataset:
         Train_Set  Val_Set  Test_Set  Total
Dataset                                     
ZENODO         460       58        58    576
FRDR           314       39        40    393


2. Split by Class:
           Train_Set  Val_Set  Test_Set  Total
Class                                         
Active           346       45        44    435
Queenless        297       36        38    371
Infested         131       16        16    163


3. Split by Hive:
           Train_Set  Val_Set  Test_Set  Total
Hive                                          
HIVE-3627          4        1         0      5
HIVE-3628         25        3         3     31
HIVE-3631         52        4         9     65
HIVE-3640         33        4         4     41
HIVE-3690         52        8         5     65
HIVE-3691         12        1         2     15
HIVE-3692          3        0         1      4
HIVE-3693 

## Slice Audio Data -> 30s

In [9]:
sr = 16000
mel_transform = T.MelSpectrogram(sample_rate=sr, n_fft=1024, hop_length=512, n_mels=128).to(DEVICE)

def bee_band_ratio(audio, sr=16000, bee_low=200, bee_high=2000):
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)
    spec = torch.stft(audio.squeeze(0), n_fft=512, hop_length=256,
                      window=torch.hann_window(512).to(audio.device),
                      return_complex=True, onesided=True)
    power = torch.abs(spec).pow(2).mean(dim=1)
    freqs = torch.linspace(0, sr/2, power.shape[0], device=audio.device)
    bee_mask = (freqs >= bee_low) & (freqs <= bee_high)
    bee_power = power[bee_mask].sum()
    total_power = power.sum()
    return (bee_power / total_power).item()

def make_mel_tensor(audio_chunk, target_size=(224, 224)):
    if audio_chunk.shape[0] > 1:
        audio_chunk = torch.mean(audio_chunk, dim=0, keepdim=True)
    mel_spec = mel_transform(audio_chunk)
    mel_db = T.AmplitudeToDB()(mel_spec)
    input_tensor = mel_db.unsqueeze(0)
    mel_resized = torch.nn.functional.interpolate(input_tensor, size=target_size, mode='bilinear', align_corners=False).squeeze(0)
    mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(3, 1, 1)
    mel_rgb = mel_resized.repeat(3, 1, 1)
    return (mel_rgb - mean) / std

def audio_chunks(df, is_train=True, chunk_length_s=30, max_eval_chunks=5):
    chunked_data = []
    chunk_len = chunk_length_s * sr
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing chunks"):
        try:
            y_full, file_sr = torchaudio.load(row['File_Path'])
            if file_sr != sr:
                resampler = T.Resample(file_sr, sr).to(DEVICE)
                y_full = resampler(y_full.to(DEVICE))
            else:
                y_full = y_full.to(DEVICE)
            total_chunks = y_full.shape[1] // chunk_len
            if total_chunks == 0: continue

            rng = np.random.default_rng(7)


            indices = range(total_chunks) if is_train else rng.choice(range(total_chunks), min(total_chunks, max_eval_chunks), replace=False)
            
            for i in indices:
                y = y_full[:, i*chunk_len : (i+1)*chunk_len]
                rms = torch.sqrt(torch.mean(y**2))
                if rms < 0.003:
                    continue
                bee_ratio = bee_band_ratio(y, sr, bee_low=200, bee_high=2000)
                if bee_ratio < 0.3:
                    continue
                melspec_tensor = make_mel_tensor(y)
                chunked_data.append({'File_Path': row['File_Path'], 'Class': row['Class'], 'chunk_id': i, 'mel_tensor': melspec_tensor.cpu()})
        except Exception as e:
            print(f"Error: {e}")
            continue
    return pd.DataFrame(chunked_data)

train_chunks_df = audio_chunks(train_df, is_train=True, chunk_length_s=30)
print(f'Train chunks: {len(train_chunks_df)} chunks {len(train_df)} files')
print('Processing validation audio chunks')
val_chunks_df = audio_chunks(val_df, is_train=False, chunk_length_s=30, max_eval_chunks=5)
print(f'Val chunks: {len(val_chunks_df)} chunks {len(val_df)} files')
print('Processing test audio chunks')
test_chunks_df = audio_chunks(test_df, is_train=False, chunk_length_s=30, max_eval_chunks=5)
print(f'Test chunks: {len(test_chunks_df)} chunks {len(test_df)} files')
print(f'\nLabel distribution in chunks')
print(f'Train: {train_chunks_df["Class"].value_counts().to_dict()}')
print(f'Val: {val_chunks_df["Class"].value_counts().to_dict()}')

Processing chunks: 100%|██████████| 774/774 [02:50<00:00,  4.53it/s]


Train chunks: 8841 chunks 774 files
Processing validation audio chunks


Processing chunks: 100%|██████████| 97/97 [00:24<00:00,  4.02it/s]


Val chunks: 225 chunks 97 files
Processing test audio chunks


Processing chunks: 100%|██████████| 98/98 [00:24<00:00,  4.03it/s]

Test chunks: 254 chunks 98 files

Label distribution in chunks
Train: {'Active': 3433, 'Queenless': 2758, 'Infested': 2650}
Val: {'Active': 95, 'Queenless': 76, 'Infested': 54}


In [10]:
def downsample(df, infested_label='Infested', other_labels=('Active', 'Queenless')):
    label_counts = df['Class'].value_counts()
    target_count = int(label_counts.get(infested_label, 0))
    if target_count == 0: return df
    balanced = []
    for label, group in df.groupby('Class'):
        if label in other_labels and len(group) > target_count:
            balanced.append(group.sample(n=target_count, random_state=42))
        else: balanced.append(group)
    balanced_df = pd.concat(balanced, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
    return balanced_df

balanced_train_chunks_df = downsample(train_chunks_df)
balanced_val_chunks_df = downsample(val_chunks_df)
print(balanced_train_chunks_df['Class'].value_counts().to_dict())
print(balanced_val_chunks_df['Class'].value_counts().to_dict())

{'Active': 2650, 'Infested': 2650, 'Queenless': 2650}
{'Queenless': 54, 'Infested': 54, 'Active': 54}


In [11]:
class WarmupSchedule:
    def __init__(self, optimizer, warmup_epochs=5, total_epochs=30, min_lr=1e-5):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        self.base_lr = optimizer.param_groups[0]['lr']
        self.epoch = 0
        for g in self.optimizer.param_groups:
            g['lr'] = self.min_lr
    def step(self):
        self.epoch += 1
        if self.epoch <= self.warmup_epochs:
            lr = self.min_lr + (self.base_lr - self.min_lr) * (self.epoch / self.warmup_epochs)
        else:
            progress = (self.epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + np.cos(np.pi * progress))
        for g in self.optimizer.param_groups:
            g['lr'] = lr
        return lr


class_weights = torch.tensor([1.0, 1.0, 3.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

## Dataset with Cutout Augmentation

In [12]:
class ChunkDataset(Dataset):
    def __init__(self, chunks_df, augment=False):
        self.chunks_df = chunks_df.reset_index(drop=True)
        self.label_map = {'Active': 0, 'Queenless': 1, 'Infested': 2}
        self.augment = augment
    def __len__(self):
        return len(self.chunks_df)
    def __getitem__(self, idx):
        row = self.chunks_df.iloc[idx]
        mel_tensor = row['mel_tensor']
        if not isinstance(mel_tensor, torch.Tensor):
            mel_tensor = torch.tensor(mel_tensor, dtype=torch.float32)
        label_str = row['Class']
        label_encoded = self.label_map.get(label_str, 0)
        if self.augment:
            if torch.rand(1) < 0.2:
                h, w = mel_tensor.shape[1], mel_tensor.shape[2]
                eh, ew = int(h*0.15), int(w*0.15)
                y = torch.randint(0, h-eh+1, (1,)).item()
                x = torch.randint(0, w-ew+1, (1,)).item()
                mel_tensor[:, y:y+eh, x:x+ew] = 0

            if torch.rand(1) < 0.2:
                f = torch.randint(0, mel_tensor.shape[1] // 5, (1,)).item()
                f0 = torch.randint(0, mel_tensor.shape[1] - f, (1,)).item()
                mel_tensor[:, f0:f0+f, :] = 0

            if torch.rand(1) < 0.2:
                t = torch.randint(0, mel_tensor.shape[2] // 5, (1,)).item()
                t0 = torch.randint(0, mel_tensor.shape[2] - t, (1,)).item()
                mel_tensor[:, :, t0:t0+t] = 0

            if torch.rand(1) < 0.1:
                noise = torch.randn_like(mel_tensor) * 0.05
                mel_tensor = mel_tensor + noise
                
        return mel_tensor, torch.tensor(label_encoded, dtype=torch.long)

pin_memory = True if torch.cuda.is_available() else False
train_dataset = ChunkDataset(balanced_train_chunks_df, augment=True)
val_dataset = ChunkDataset(balanced_val_chunks_df)
test_dataset = ChunkDataset(test_chunks_df)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=pin_memory)

## Train: MobileNetV4 Hybrid Large

In [ ]:
model = timm.create_model('mobilenetv4_hybrid_large', pretrained=True, num_classes=3)
in_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4), nn.Linear(in_features, 3)
)
print(model.classifier)
mobilenet = model.to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

NUM_EPOCHS = 30
scheduler = WarmupSchedule(optimizer, warmup_epochs=8, total_epochs=NUM_EPOCHS)

best_val_loss = float('inf')
patience, trigger = 10, 0
results_table = []
headers = ["Epoch", "Train Loss", "Train Acc", "Val Loss", "Val Acc", "LR", "Grad Norm", "Time"]

for epoch in range(NUM_EPOCHS):
    mobilenet.train()
    start_time = time.time()
    running_loss, correct, total = 0.0, 0, 0
    running_grad_norm = 0.0
    num_batches = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)

    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = mobilenet(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(mobilenet.parameters(), max_norm=3.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        running_grad_norm += grad_norm.item()
        num_batches += 1
        train_loop.set_postfix(loss=f"{loss.item():.4f}")

    train_loss, train_acc = running_loss / total, correct / total
    avg_grad_norm = running_grad_norm / num_batches if num_batches > 0 else 0

    mobilenet.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = mobilenet(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss, val_acc = val_loss / val_total, val_correct / val_total
    lr_val = scheduler.step()

    results_table.append([epoch+1, f"{train_loss:.4f}", f"{train_acc:.4f}", f"{val_loss:.4f}", f"{val_acc:.4f}", f"{lr_val:.2e}", f"{avg_grad_norm:.4f}", f"{time.time() - start_time:.1f}s"])
    clear_output(wait=True)
    print(tabulate(results_table, headers=headers, tablefmt="simple_outline"))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(mobilenet.state_dict(), 'models/best_mobilenet_model.pth')
        trigger = 0
        print(f"    Saved best model at epoch {epoch+1}")
    else:
        trigger += 1
        if trigger >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

mobilenet.load_state_dict(torch.load('models/best_mobilenet_model.pth'))
print('Loaded best model weights')

## Train: MobileNetV3 Large


In [ ]:
model_name = "mobilenetv3_large_100"
model = timm.create_model(model_name, pretrained=True)
in_features = model.classifier.in_features
model.classifier = nn.Sequential(nn.Dropout(0.6), nn.Linear(in_features, 3))

model = model.to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=3e-2)

NUM_EPOCHS = 30
scheduler = WarmupSchedule(optimizer, warmup_epochs=5, total_epochs=NUM_EPOCHS)

best_val_loss = float("inf")
patience, trigger = 7, 0
results_table = []
headers = ["Epoch", "Train Loss", "Train Acc", "Val Loss", "Val Acc", "LR", "Grad Norm", "Time"]

for epoch in range(NUM_EPOCHS):
    model.train()
    start_time = time.time()
    running_loss, correct, total = 0.0, 0, 0
    running_grad_norm = 0.0
    num_batches = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)

    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        running_grad_norm += grad_norm.item()
        num_batches += 1
        train_loop.set_postfix(loss=f"{loss.item():.4f}")

    train_loss, train_acc = running_loss / total, correct / total
    avg_grad_norm = running_grad_norm / num_batches if num_batches > 0 else 0

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss, val_acc = val_loss / val_total, val_correct / val_total
    
    lr_val = scheduler.step()

    ts = time.time() - start_time
    results_table.append([epoch+1, f"{train_loss:.4f}", f"{train_acc:.4f}", f"{val_loss:.4f}", f"{val_acc:.4f}", f"{lr_val:.2e}", f"{avg_grad_norm:.4f}", f"{ts:.1f}s"])
    clear_output(wait=True)
    print(tabulate(results_table, headers=headers, tablefmt="simple_outline"))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"models/best_{model_name}_model.pth")
        trigger = 0
        print(f"    Saved best {model_name} at epoch {epoch+1}")
    else:
        trigger += 1
        if trigger >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(f"models/best_{model_name}_model.pth"))
print(f"Loaded best {model_name} weights")

## Train: ConvNeXt Tiny


In [ ]:
model_name = "convnext_tiny"
model = timm.create_model(model_name, pretrained=True)
in_features = model.head.fc.in_features
model.head.fc = nn.Linear(in_features, 3)
model.head.drop = nn.Dropout(0.4)
model = model.to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

NUM_EPOCHS = 30
scheduler = WarmupSchedule(optimizer, warmup_epochs=8, total_epochs=NUM_EPOCHS)

best_val_loss = float("inf")
patience, trigger = 7, 0
results_table = []
headers = ["Epoch", "Train Loss", "Train Acc", "Val Loss", "Val Acc", "LR", "Grad Norm", "Time"]

for epoch in range(NUM_EPOCHS):
    model.train()
    start_time = time.time()
    running_loss, correct, total = 0.0, 0, 0
    running_grad_norm = 0.0
    num_batches = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)

    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        running_grad_norm += grad_norm.item()
        num_batches += 1
        train_loop.set_postfix(loss=f"{loss.item():.4f}")

    train_loss, train_acc = running_loss / total, correct / total
    avg_grad_norm = running_grad_norm / num_batches if num_batches > 0 else 0

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss, val_acc = val_loss / val_total, val_correct / val_total
    lr_val = scheduler.step()

    ts = time.time() - start_time
    results_table.append([epoch+1, f"{train_loss:.4f}", f"{train_acc:.4f}", f"{val_loss:.4f}", f"{val_acc:.4f}", f"{lr_val:.2e}", f"{avg_grad_norm:.4f}", f"{ts:.1f}s"])
    clear_output(wait=True)
    print(tabulate(results_table, headers=headers, tablefmt="simple_outline"))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"models/best_{model_name}_model.pth")
        trigger = 0
        print(f"    Saved best {model_name} at epoch {epoch+1}")
    else:
        trigger += 1
        if trigger >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(f"models/best_{model_name}_model.pth"))
print(f"Loaded best {model_name} weights")

## Train: EfficientNet-B0


In [ ]:
model_name = "efficientnet_b0"
model = timm.create_model(model_name, pretrained=True)
in_features = model.classifier.in_features
model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_features, 3))
model = model.to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=3e-2)

NUM_EPOCHS = 30
scheduler = WarmupSchedule(optimizer, warmup_epochs=8, total_epochs=NUM_EPOCHS)

best_val_loss = float("inf")
patience, trigger = 7, 0
results_table = []
headers = ["Epoch", "Train Loss", "Train Acc", "Val Loss", "Val Acc", "LR", "Grad Norm", "Time"]

for epoch in range(NUM_EPOCHS):
    model.train()

    start_time = time.time()
    running_loss, correct, total = 0.0, 0, 0
    running_grad_norm = 0.0
    num_batches = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)

    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        running_grad_norm += grad_norm.item()
        num_batches += 1
        train_loop.set_postfix(loss=f"{loss.item():.4f}")

    train_loss, train_acc = running_loss / total, correct / total
    avg_grad_norm = running_grad_norm / num_batches if num_batches > 0 else 0

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss, val_acc = val_loss / val_total, val_correct / val_total
    lr_val = scheduler.step()

    ts = time.time() - start_time
    results_table.append([epoch+1, f"{train_loss:.4f}", f"{train_acc:.4f}", f"{val_loss:.4f}", f"{val_acc:.4f}", f"{lr_val:.2e}", f"{avg_grad_norm:.4f}", f"{ts:.1f}s"])
    clear_output(wait=True)
    print(tabulate(results_table, headers=headers, tablefmt="simple_outline"))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"models/best_{model_name}_model.pth")
        trigger = 0
        print(f"    Saved best {model_name} at epoch {epoch+1}")
    else:
        trigger += 1
        if trigger >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(f"models/best_{model_name}_model.pth"))
print(f"Loaded best {model_name} weights")

In [ ]:
results_df = pd.DataFrame(results_table, columns=headers)
cols = ['Train Loss', 'Train Acc', 'Val Loss', 'Val Acc']
for c in cols:
    results_df[c] = results_df[c].astype(float)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].plot(results_df['Epoch'], results_df['Train Loss'], label='Train')
axs[0].plot(results_df['Epoch'], results_df['Val Loss'], label='Val')
axs[0].set_title('Loss'); axs[0].legend()
axs[1].plot(results_df['Epoch'], results_df['Train Acc'], label='Train')
axs[1].plot(results_df['Epoch'], results_df['Val Acc'], label='Val')
axs[1].set_title('Accuracy'); axs[1].legend()

best_idx = results_df['Val Loss'].idxmin()
plt.tight_layout(); plt.show()
print("Total Epochs:", len(results_df))
print("Best Epoch:", results_df.loc[best_idx, 'Epoch'])
print("Best val loss:", results_df.loc[best_idx, 'Val Loss'])

In [ ]:
def plot_density(model, loader, device, class_names=['Active', 'Queenless', 'Infested']):
    model.eval()
    all_probs = []; all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    probs = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for i in range(3):
        for j in range(3):
            sns.kdeplot(probs[labels == j, i], fill=True, color=colors[j], label=f"Actual {class_names[j]}", ax=axes[i], alpha=0.3, linewidth=2)
        axes[i].set_title(f"Confidence score for {class_names[i]}")
        axes[i].set_xlabel("Probability"); axes[i].set_ylabel("Density"); axes[i].set_xlim(-0.05, 1.05); axes[i].legend()
    plt.tight_layout(); plt.show()

plot_density(model, test_loader, DEVICE)

In [ ]:
results_df = pd.DataFrame(results_table, columns=headers)
results_df['LR'] = results_df['LR'].astype(float)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results_df['Epoch'], results_df['LR'], marker='o', linestyle='-', color='green', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
def plot_metrics(model, dataloader, device, class_names, split_name=None):
    model.eval()
    all_preds = []; all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
    print(classification_report(all_labels, all_preds, target_names=class_names))
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax, cbar_kws={'label': 'Count'})
    ax.set_xlabel('Predicted Labels', fontsize=11, labelpad=10)
    ax.set_ylabel('True Labels', fontsize=11, labelpad=10)
    title = 'Confusion Matrix'
    if split_name: title += f" ({split_name})"
    ax.set_title(title, fontsize=13, pad=15)
    plt.tight_layout(); plt.show()
    return all_preds, all_labels

class_labels = ['Active', 'Queenless', 'Infested']
val_preds, val_labels = plot_metrics(model, val_loader, DEVICE, class_labels, split_name="Validation Set")

# Model Comparison

In [28]:
def _build_model(arch_name, dropout_rate=0.4):
    model = timm.create_model(arch_name, pretrained=False, num_classes=3)
    cls = model.classifier
    in_feats = cls.in_features if isinstance(cls, nn.Linear) else next(l.in_features for l in reversed(list(cls)) if isinstance(l, nn.Linear))
    model.classifier = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(in_feats, 3))
    return model

def _build_convnext(dropout_rate=0.4):
    model = timm.create_model('convnext_tiny', pretrained=False)
    model.head.fc = nn.Linear(model.head.fc.in_features, 3)
    model.head.drop = nn.Dropout(dropout_rate)
    return model

DISPLAY_NAMES = {'mobilenetv4_hybrid_large': 'MobileNetV4 Hybrid Large', 'mobilenetv3_large_100': 'MobileNetV3 Large 100', 'convnext_tiny': 'ConvNeXt Tiny', 'efficientnet_b0': 'EfficientNet-B0', 'efficientnet_b1': 'EfficientNet-B1', 'efficientnet_b2': 'EfficientNet-B2'}
MODEL_REGISTRY = []
for filepath in sorted(glob.glob('models/save_best_*.pth')):
    stem = os.path.basename(filepath).replace('save_best_', '').replace('_model.pth', '')
    arch_name = {'mobilenetv4': 'mobilenetv4_hybrid_large'}.get(stem, stem)
    MODEL_REGISTRY.append((filepath, DISPLAY_NAMES.get(arch_name, stem.replace('_', ' ').title()), _build_convnext if arch_name == 'convnext_tiny' else (lambda a=arch_name: _build_model(a))))

results = []
for filepath, label, builder in MODEL_REGISTRY:
    model = builder()
    model.load_state_dict(torch.load(filepath, map_location=DEVICE, weights_only=True))
    model = model.to(DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            _, pred = model(imgs.to(DEVICE)).max(1)
            preds.extend(pred.cpu().tolist())
            labels.extend(lbls.tolist())
    acc = (np.array(preds) == np.array(labels)).mean()
    report = classification_report(labels, preds, target_names=['Active', 'Queenless', 'Infested'], output_dict=True)
    results.append({'Model': label, 'Params': f'{n_params:.1f}M', 'Accuracy': acc,
                    'Precision': report['macro avg']['precision'], 'Recall': report['macro avg']['recall'],
                    'F1': report['macro avg']['f1-score'], 'Active F1': report['Active']['f1-score'],
                    'Queenless F1': report['Queenless']['f1-score'], 'Infested F1': report['Infested']['f1-score']})

df = pd.DataFrame(results)
display_cols = ['Model', 'Params', 'Accuracy', 'Precision', 'Recall', 'F1', 'Active F1', 'Queenless F1', 'Infested F1']
display(df[display_cols].style.format({'Accuracy': '{:.2%}', 'Precision': '{:.3f}', 'Recall': '{:.3f}', 'F1': '{:.3f}', 'Active F1': '{:.3f}', 'Queenless F1': '{:.3f}', 'Infested F1': '{:.3f}'}))

,Model,Params,Accuracy,Precision,Recall,F1,Active F1,Queenless F1,Infested F1
0,ConvNeXt Tiny,27.8M,92.13%,0.919,0.928,0.918,0.886,0.989,0.878
1,EfficientNet-B0,4.0M,83.86%,0.846,0.851,0.831,0.742,0.958,0.792
2,EfficientNet-B1,6.5M,92.91%,0.923,0.931,0.926,0.902,0.989,0.886
3,EfficientNet-B2,7.7M,85.43%,0.852,0.865,0.852,0.784,0.921,0.851
4,MobileNetV3 Large 100,4.2M,83.46%,0.840,0.849,0.828,0.738,0.937,0.810
5,MobileNetV4 Hybrid Large,36.5M,94.88%,0.944,0.953,0.946,0.928,0.995,0.915
